# 10 — Noise Robustness (Yeast GI PCC)

```text
Reviewer concern addressed: R1-b -- "enrichment depends on dendrogram cut; comment on
    stability under noise / across settings." Implements frozen-plan Phase 2 notebook 10,
    the first of four Go/No-Go-checkpoint inputs (10, 11, 20, 21).
Input files: data/yeast/gi_pcc_sampled.tsv, data/yeast/go_bp_name_to_orfs.json
    (hashes cross-checked against 00_environment_check_manifest.json)
HiMaLAYAS version: verified git tag v0.0.15, activated via
    revision_utils.activate_pinned_source() -- the same pattern 01 established. The active
    kernel's editable 0.0.16a0 install is NOT used for any reported result in this notebook.
Random seed: DEFAULT_RANDOM_SEED=0 (revision_utils.seeds); five replicate seeds derived as
    [0, 1, 2, 3, 4], one per seeded noise replicate at each noise level. The frac=0.0
    reference run is deterministic and unseeded (matches fig_1.ipynb exactly).
Primary parameters: reference clustering/enrichment config reproduced verbatim from
    fig_1.ipynb (linkage_method="ward", linkage_metric="euclidean", linkage_threshold=16,
    optimal_ordering=True, min_cluster_size=30, min_overlap=2, qval<=0.05). Noise grid:
    symmetric Gaussian noise at fractions [0.05, 0.10, 0.20, 0.30, 0.50] of the matrix's
    off-diagonal standard deviation, 5 seeded replicates per level (25 perturbed runs total).
Outputs written:
    revision/outputs/manifests/10_noise_robustness_gi_pcc_manifest.json
    revision/outputs/tables/10_noise_robustness_gi_pcc/*.csv (5 tables)
    revision/outputs/figures/10_noise_robustness_gi_pcc/*.png (2 figures)
Interpretation: see the Go / Conditional Go / Concern decision in the final section.
```

## Design

fig_1.ipynb's clustering treats each gene's row of the 1053x1053 GI-profile-similarity
matrix as a 1053-dimensional feature vector and runs Ward/Euclidean linkage directly on
those vectors (confirmed by reading `himalayas/core/clustering.py`: `compute_linkage` calls
`scipy.cluster.hierarchy.linkage(matrix.values, ...)`). Perturbing matrix *values* therefore
perturbs the actual clustering input, not just a downstream visualization.

Cluster ids are **not comparable** across independently reclustered noisy replicates -- a
"cluster 3" in one run has no necessary relationship to "cluster 3" in another. Per the
frozen plan (Section 9, Frozen Non-Goals) this notebook does not attempt a differential
cluster-identity comparison; instead, each reference cluster is matched to its
best-overlapping cluster in every perturbed replicate by **gene-membership Jaccard
similarity**, and stability is assessed on that matched pair. This is the "cluster-label or
cluster-membership similarity where feasible" metric called for in the notebook brief.

## 1. Locate repo root and import shared revision utilities

In [1]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    markers = ("himalayas_src", "data", ".git", "revision")
    for candidate in (start, *start.parents):
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(f"Could not locate himalayas-publication repo root above {start}")


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "revision" / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Kernel CWD:  {Path.cwd()}")
print(f"Repo root:   {REPO_ROOT}")

Kernel CWD:  /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/notebooks
Repo root:   /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication


In [2]:
import json
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import revision_utils as ru
from revision_utils import DEFAULT_RANDOM_SEED, revision_layout

layout = revision_layout(REPO_ROOT)
for key in ("manifests_dir", "figures_dir", "tables_dir", "scratch_dir"):
    layout[key].mkdir(parents=True, exist_ok=True)

NB_ID = "10_noise_robustness_gi_pcc"
NB_FIGURES_DIR = layout["figures_dir"] / NB_ID
NB_TABLES_DIR = layout["tables_dir"] / NB_ID
NB_SCRATCH_DIR = layout["scratch_dir"] / NB_ID
for d in (NB_FIGURES_DIR, NB_TABLES_DIR, NB_SCRATCH_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Figures ->", NB_FIGURES_DIR)
print("Tables  ->", NB_TABLES_DIR)
print("Scratch ->", NB_SCRATCH_DIR)

Figures -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/figures/10_noise_robustness_gi_pcc
Tables  -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/10_noise_robustness_gi_pcc
Scratch -> /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/scratch/10_noise_robustness_gi_pcc


## 2. Recall notebooks 00/01 findings and re-verify the environment now

In [3]:
nb00_manifest = ru.read_manifest(layout["manifests_dir"] / "00_environment_check_manifest.json")
nb01_manifest = ru.read_manifest(
    layout["manifests_dir"] / "01_reproduce_submitted_figures_manifest.json"
)

print("00_environment_check flags:")
for f in nb00_manifest["flags"]:
    print(f"  - {f}")
print()
print(
    f"01_reproduce_submitted_figures status: {nb01_manifest['run_records'][0]['success']=} "
    f"(baseline definition: {nb01_manifest['baseline_policy']['baseline_definition'][:80]}...)"
)

himalayas_diag_before = ru.himalayas_diagnostics(layout["repo_root"])
print()
print(
    f"Active kernel HiMaLAYAS (before activation below): {himalayas_diag_before['imported_version']} "
    f"(editable={himalayas_diag_before['is_editable_install']})"
)

00_environment_check flags:
  - ACTIVE HiMaLAYAS VERSION MISMATCH: kernel imports '0.0.16a0' but README pins '0.0.15'.
  - EDITABLE HiMaLAYAS SOURCE IS DIRTY: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas has 5 uncommitted path(s): ['M src/himalayas/__init__.py', '?? PREVIEW_NOTES.md', '?? _archive/', '?? preview-release-notes.sh', '?? test.py'].
  - EDITABLE HiMaLAYAS SOURCE IS AHEAD OF ITS LAST TAG: git describe = 'v0.0.15-4-g0c51115'.
  - VENDORED COPY VERSION DIFFERS FROM IMPORTED VERSION: himalayas_src/ declares '0.0.15', kernel imports '0.0.16a0'.
  - PUBLICATION REPO HAS UNCOMMITTED/UNTRACKED PATHS: 6 path(s) -- expected during active revision work (e.g. this new revision/ tree itself); noted for provenance only.

01_reproduce_submitted_figures status: nb01_manifest['run_records'][0]['success']=True (baseline definition: HiMaLAYAS source at git tag v0.0.15 in the upstream package repository, verified...)



Active kernel HiMaLAYAS (before activation below): 0.0.16a0 (editable=True)


## 3. Activate the verified pinned v0.0.15 source

Following notebook 01's explicit recommendation: continue importing HiMaLAYAS via the verified-tag extraction approach rather than the active editable install or the (known-to-diverge) vendored `himalayas_src/` copy, until that discrepancy is resolved.

In [4]:
PINNED_TAG = "v0.0.15"

try:
    activation = ru.activate_pinned_source(
        layout["repo_root"], NB_SCRATCH_DIR / "himalayas_pinned_v0_0_15", tag=PINNED_TAG
    )
    activation_error = None
except RuntimeError as exc:
    activation = None
    activation_error = str(exc)

if activation is not None:
    print(json.dumps(activation, indent=2))
else:
    print(f"ACTIVATION FAILED: {activation_error}")

import himalayas
from himalayas import Analysis, Annotations, Matrix

print()
print(f"himalayas module now bound to: {himalayas.__file__}")
print(f"himalayas.__version__:         {himalayas.__version__}")

if activation is not None:
    assert (
        himalayas.__version__ == "0.0.15"
    ), f"Expected pinned v0.0.15 after activation, got {himalayas.__version__!r}"
    assert activation["matches_requested_tag"], "activate_pinned_source reported a version mismatch"
    print("\nConfirmed: this notebook's HiMaLAYAS analyses run under verified tag v0.0.15.")

{
  "requested_tag": "v0.0.15",
  "sibling_repo": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas",
  "pinned_src_dir": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/scratch/10_noise_robustness_gi_pcc/himalayas_pinned_v0_0_15/src",
  "activated_version": "0.0.15",
  "activated_file": "/Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/scratch/10_noise_robustness_gi_pcc/himalayas_pinned_v0_0_15/src/himalayas/__init__.py",
  "matches_requested_tag": true
}

himalayas module now bound to: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/scratch/10_noise_robustness_gi_pcc/himalayas_pinned_v0_0_15/src/himalayas/__init__.py
himalayas.__version__:         0.0.15

Confirmed: this notebook's HiMaLAYAS analyses run under verified tag v0.0.15.


## 4. Load inputs and cross-check hashes against notebook 00

In [5]:
GI_PCC_PATH = layout["repo_root"] / "data" / "yeast" / "gi_pcc_sampled.tsv"
GO_BP_PATH = layout["repo_root"] / "data" / "yeast" / "go_bp_name_to_orfs.json"

current_hashes = {
    "data/yeast/gi_pcc_sampled.tsv": ru.sha256_file(GI_PCC_PATH),
    "data/yeast/go_bp_name_to_orfs.json": ru.sha256_file(GO_BP_PATH),
}
recorded_hashes = {r["path"]: r["sha256"] for r in nb00_manifest["input_files"]}

input_hash_flags = []
for rel_path, current_hash in current_hashes.items():
    recorded_hash = recorded_hashes.get(rel_path)
    matches = current_hash == recorded_hash
    print(f"{rel_path}: matches 00_environment_check record = {matches}")
    if not matches:
        input_hash_flags.append(
            f"INPUT HASH MISMATCH: {rel_path} changed since 00_environment_check ran "
            f"(recorded {recorded_hash}, now {current_hash})."
        )

DF_GI_PCC = pd.read_csv(GI_PCC_PATH, sep="\t", index_col=0)
with open(GO_BP_PATH, "r", encoding="utf-8") as fh:
    go_bp = json.load(fh)

print()
print(f"Matrix shape: {DF_GI_PCC.shape}")
offdiag_mask = ~np.eye(DF_GI_PCC.shape[0], dtype=bool)
offdiag_std = float(DF_GI_PCC.values[offdiag_mask].std())
print(f"Off-diagonal std (noise-scale reference): {offdiag_std:.6f}")
print(f"GO BP terms loaded: {len(go_bp):,}")

data/yeast/gi_pcc_sampled.tsv: matches 00_environment_check record = True
data/yeast/go_bp_name_to_orfs.json: matches 00_environment_check record = True

Matrix shape: (1053, 1053)
Off-diagonal std (noise-scale reference): 0.068718
GO BP terms loaded: 1,095


## 5. Reference analysis (no perturbation)

Reproduces fig_1.ipynb's exact clustering/enrichment configuration and cross-checks the result against notebook 01's recorded reproduction (709 / 331 / 7).

In [6]:
REFERENCE_PARAMS = dict(
    linkage_method="ward",
    linkage_metric="euclidean",
    linkage_threshold=16,
    optimal_ordering=True,
    min_cluster_size=30,
)
MIN_OVERLAP = 2
QVAL_CUTOFF = 0.05

with warnings.catch_warnings():
    warnings.simplefilter(
        "ignore", RuntimeWarning
    )  # expected: GO terms outside the matrix universe
    reference_matrix = Matrix(DF_GI_PCC)
    reference_annotations = Annotations(go_bp, reference_matrix)
    reference_analysis = (
        Analysis(reference_matrix, reference_annotations)
        .cluster(**REFERENCE_PARAMS)
        .enrich(min_overlap=MIN_OVERLAP)
        .finalize(col_cluster=True)
    )

reference_results = reference_analysis.results
reference_results_sig = reference_results.filter(f"qval <= {QVAL_CUTOFF}")
reference_cluster_labels = reference_results_sig.cluster_labels(rank_by="p", label_mode="top_term")
reference_cluster_to_labels = {
    int(cid): set(genes) for cid, genes in reference_analysis.clusters.cluster_to_labels.items()
}

print(
    f"Reference: {len(reference_results.df)} enriched rows, "
    f"{len(reference_results_sig.df)} significant (q<={QVAL_CUTOFF}), "
    f"{len(reference_cluster_labels)} labeled clusters"
)

expected = nb01_manifest["cross_mode_comparison"][0]  # fig_1.ipynb, pinned_v0_0_15 vs dev
reproduction_matches = (
    len(reference_results.df) == expected["results_rows_baseline"]
    and len(reference_results_sig.df) == expected["results_sig_rows_baseline"]
    and len(reference_cluster_labels) == expected["cluster_labels_rows_baseline"]
)
print(f"\nMatches notebook 01's recorded fig_1.ipynb baseline (709/331/7): {reproduction_matches}")
assert (
    reproduction_matches
), "Reference analysis does not match the notebook-01 baseline -- stop and investigate"

reference_cluster_labels

Reference: 709 enriched rows, 331 significant (q<=0.05), 7 labeled clusters

Matches notebook 01's recorded fig_1.ipynb baseline (709/331/7): True


,cluster,label,pval,qval,score,n,term,fe
0,1,GPI anchor biosynthetic process,3.222730e-12,8.788137e-11,3.222730e-12,148,GPI anchor biosynthetic process,5.568155
1,2,vesicle-mediated transport,1.237408e-26,4.386613e-24,1.237408e-26,92,vesicle-mediated transport,7.868886
2,3,"mRNA splicing, via spliceosome",2.829998e-16,1.433192e-14,2.829998e-16,263,"mRNA splicing, via spliceosome",3.745492
3,4,cytoplasmic translation,8.925220e-15,3.954988e-13,8.925220e-15,358,cytoplasmic translation,2.852209
4,5,mitochondrial respiratory chain complex IV ass...,1.794876e-19,1.590709e-17,1.794876e-19,78,mitochondrial respiratory chain complex IV ass...,12.750000
5,6,DNA replication,1.798541e-42,1.275166e-39,1.798541e-42,52,DNA replication,16.011628
6,7,cell division,2.138848e-20,2.707850e-18,2.138848e-20,62,cell division,8.177419


## 6. Noise perturbation design and seed policy

In [7]:
NOISE_FRACS = [0.05, 0.10, 0.20, 0.30, 0.50]  # fractions of offdiag_std
N_SEEDS = 5
REPLICATE_SEEDS = [DEFAULT_RANDOM_SEED + i for i in range(N_SEEDS)]
STABLE_RETENTION_THRESHOLD = 0.8
MODERATE_NOISE_CEILING = 0.20  # highest frac still counted as "moderate" perturbation

print(f"Noise fractions (of offdiag_std={offdiag_std:.6f}): {NOISE_FRACS}")
print(
    f"Replicate seeds (derived from DEFAULT_RANDOM_SEED={DEFAULT_RANDOM_SEED}): {REPLICATE_SEEDS}"
)
print(
    f"Total perturbed runs: {len(NOISE_FRACS)} levels x {N_SEEDS} seeds = "
    f"{len(NOISE_FRACS) * N_SEEDS}, plus 1 deterministic reference (frac=0.0)"
)
print(f"Stable-annotation retention threshold: {STABLE_RETENTION_THRESHOLD}")
print(
    f"'Moderate perturbation' ceiling for the Go/Conditional-Go/Concern decision: "
    f"frac <= {MODERATE_NOISE_CEILING}"
)
print(
    "Diagonal handling: noise is applied uniformly to the full 1053x1053 matrix via "
    "revision_utils.symmetric_noise, including diagonal entries (i==j) -- there is no "
    "special-casing that leaves the diagonal unperturbed."
)

Noise fractions (of offdiag_std=0.068718): [0.05, 0.1, 0.2, 0.3, 0.5]
Replicate seeds (derived from DEFAULT_RANDOM_SEED=0): [0, 1, 2, 3, 4]
Total perturbed runs: 5 levels x 5 seeds = 25, plus 1 deterministic reference (frac=0.0)
Stable-annotation retention threshold: 0.8
'Moderate perturbation' ceiling for the Go/Conditional-Go/Concern decision: frac <= 0.2
Diagonal handling: noise is applied uniformly to the full 1053x1053 matrix via revision_utils.symmetric_noise, including diagonal entries (i==j) -- there is no special-casing that leaves the diagonal unperturbed.


## 7. Perturbation loop

For each (noise level, seed): perturb the matrix with symmetric noise, recluster and re-enrich under the identical reference parameters (`col_cluster=False` since no matrix is plotted here -- this does not affect clustering or enrichment), match each reference cluster to its best gene-membership overlap in the perturbed clustering, and record whether the reference cluster's headline (top) term survives at q<=0.05 in the matched perturbed cluster.

In [8]:
def cluster_term_sets(results_sig_df: pd.DataFrame) -> dict:
    """cluster_id -> set of significant term names, from a results_sig.df."""
    if results_sig_df.empty:
        return {}
    return {int(cid): set(sub["term"]) for cid, sub in results_sig_df.groupby("cluster")}


reference_term_sets = cluster_term_sets(reference_results_sig.df)
reference_global_sig_terms = set(reference_results_sig.df["term"])
reference_top_term_by_cluster = dict(
    zip(reference_cluster_labels["cluster"], reference_cluster_labels["term"])
)

cluster_rows = []
global_rows = []

for frac in NOISE_FRACS:
    level_cluster_rows = []
    level_global_rows = []
    for seed in REPLICATE_SEEDS:
        rng = np.random.default_rng(seed)
        noise = ru.symmetric_noise(DF_GI_PCC.shape, frac * offdiag_std, rng)
        perturbed_values = DF_GI_PCC.values + noise
        perturbed_df = pd.DataFrame(
            perturbed_values, index=DF_GI_PCC.index, columns=DF_GI_PCC.columns
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            p_matrix = Matrix(perturbed_df)
            p_annotations = Annotations(go_bp, p_matrix)
            p_analysis = (
                Analysis(p_matrix, p_annotations)
                .cluster(**REFERENCE_PARAMS)
                .enrich(min_overlap=MIN_OVERLAP)
                .finalize(col_cluster=False)
            )
        p_results = p_analysis.results
        p_results_sig = p_results.filter(f"qval <= {QVAL_CUTOFF}")
        p_cluster_to_labels = {
            int(cid): set(genes) for cid, genes in p_analysis.clusters.cluster_to_labels.items()
        }
        p_term_sets = cluster_term_sets(p_results_sig.df)
        p_global_sig_terms = set(p_results_sig.df["term"])
        p_cluster_labels_df = p_results_sig.cluster_labels(rank_by="p", label_mode="top_term")
        p_top_term_by_cluster = dict(
            zip(p_cluster_labels_df["cluster"], p_cluster_labels_df["term"])
        )

        matches = ru.match_clusters_by_membership(reference_cluster_to_labels, p_cluster_to_labels)

        for ref_cid, match in matches.items():
            matched_cid = match["matched_cluster_id"]
            ref_terms = reference_term_sets.get(ref_cid, set())
            matched_terms = (
                p_term_sets.get(matched_cid, set()) if matched_cid is not None else set()
            )
            ref_top_term = reference_top_term_by_cluster.get(ref_cid)
            level_cluster_rows.append(
                {
                    "noise_frac": frac,
                    "seed": seed,
                    "ref_cluster_id": ref_cid,
                    "ref_top_term": ref_top_term,
                    "ref_n_genes": len(reference_cluster_to_labels[ref_cid]),
                    "matched_cluster_id": matched_cid,
                    "membership_jaccard": match["membership_jaccard"],
                    "cluster_term_jaccard": ru.jaccard(ref_terms, matched_terms),
                    "top_term_retained": bool(ref_top_term in matched_terms),
                    "matched_top_term": p_top_term_by_cluster.get(matched_cid),
                }
            )

        level_global_rows.append(
            {
                "noise_frac": frac,
                "seed": seed,
                "n_clusters_perturbed": len(p_analysis.clusters.unique_clusters),
                "sig_rows_perturbed": len(p_results_sig.df),
                "global_sig_term_jaccard": ru.jaccard(
                    reference_global_sig_terms, p_global_sig_terms
                ),
                "global_sig_term_retention": (
                    len(reference_global_sig_terms & p_global_sig_terms)
                    / len(reference_global_sig_terms)
                    if reference_global_sig_terms
                    else None
                ),
            }
        )

    cluster_rows.extend(level_cluster_rows)
    global_rows.extend(level_global_rows)

    level_df = pd.DataFrame(level_cluster_rows)
    level_global_df = pd.DataFrame(level_global_rows)
    print(
        f"frac={frac:.2f}  top-term retention={level_df['top_term_retained'].mean():.2f}  "
        f"mean membership Jaccard={level_df['membership_jaccard'].mean():.3f}  "
        f"mean n_clusters={level_global_df['n_clusters_perturbed'].mean():.1f}  "
        f"mean global sig-term Jaccard={level_global_df['global_sig_term_jaccard'].mean():.3f}"
    )

per_replicate_cluster_raw = pd.DataFrame(cluster_rows)
per_replicate_global_raw = pd.DataFrame(global_rows)
print(
    f"\nCollected {len(per_replicate_cluster_raw)} cluster-level records and "
    f"{len(per_replicate_global_raw)} replicate-level records."
)

frac=0.05  top-term retention=1.00  mean membership Jaccard=0.816  mean n_clusters=6.6  mean global sig-term Jaccard=0.831


frac=0.10  top-term retention=0.97  mean membership Jaccard=0.701  mean n_clusters=6.2  mean global sig-term Jaccard=0.784


frac=0.20  top-term retention=0.97  mean membership Jaccard=0.665  mean n_clusters=6.4  mean global sig-term Jaccard=0.774


frac=0.30  top-term retention=1.00  mean membership Jaccard=0.660  mean n_clusters=6.2  mean global sig-term Jaccard=0.753


frac=0.50  top-term retention=0.97  mean membership Jaccard=0.682  mean n_clusters=6.2  mean global sig-term Jaccard=0.781

Collected 175 cluster-level records and 25 replicate-level records.


## 8. Aggregate: stability summary and retained top-term tables

In [9]:
stability_summary = (
    per_replicate_global_raw.groupby("noise_frac")
    .agg(
        n_replicates=("seed", "count"),
        mean_n_clusters_perturbed=("n_clusters_perturbed", "mean"),
        mean_sig_rows_perturbed=("sig_rows_perturbed", "mean"),
        mean_global_sig_term_jaccard=("global_sig_term_jaccard", "mean"),
        mean_global_sig_term_retention=("global_sig_term_retention", "mean"),
    )
    .reset_index()
)
cluster_agg = (
    per_replicate_cluster_raw.groupby("noise_frac")
    .agg(
        mean_membership_jaccard=("membership_jaccard", "mean"),
        min_membership_jaccard=("membership_jaccard", "min"),
        mean_cluster_term_jaccard=("cluster_term_jaccard", "mean"),
        top_term_retention_rate=("top_term_retained", "mean"),
    )
    .reset_index()
)
stability_summary = stability_summary.merge(cluster_agg, on="noise_frac").sort_values("noise_frac")

reference_row = pd.DataFrame(
    [
        {
            "noise_frac": 0.0,
            "n_replicates": 1,
            "mean_n_clusters_perturbed": len(reference_analysis.clusters.unique_clusters),
            "mean_sig_rows_perturbed": len(reference_results_sig.df),
            "mean_global_sig_term_jaccard": 1.0,
            "mean_global_sig_term_retention": 1.0,
            "mean_membership_jaccard": 1.0,
            "min_membership_jaccard": 1.0,
            "mean_cluster_term_jaccard": 1.0,
            "top_term_retention_rate": 1.0,
        }
    ]
)
stability_summary = pd.concat([reference_row, stability_summary], ignore_index=True)
stability_summary

,noise_frac,n_replicates,mean_n_clusters_perturbed,mean_sig_rows_perturbed,mean_global_sig_term_jaccard,mean_global_sig_term_retention,mean_membership_jaccard,min_membership_jaccard,mean_cluster_term_jaccard,top_term_retention_rate
0,0.00,1,7.0,331.0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,0.05,5,6.6,301.6,0.830623,0.869136,0.816097,0.335484,0.767751,1.000000
2,0.10,5,6.2,295.6,0.783577,0.837037,0.700784,0.325000,0.654515,0.971429
3,0.20,5,6.4,295.0,0.774251,0.829630,0.665291,0.281437,0.641069,0.971429
4,0.30,5,6.2,285.6,0.752875,0.805556,0.660495,0.315152,0.641787,1.000000
5,0.50,5,6.2,296.8,0.780764,0.837654,0.681783,0.307692,0.660962,0.971429


In [10]:
retained_top_term_summary = (
    per_replicate_cluster_raw.groupby(["ref_cluster_id", "noise_frac"])
    .agg(
        ref_top_term=("ref_top_term", "first"),
        ref_n_genes=("ref_n_genes", "first"),
        n_replicates=("seed", "count"),
        top_term_retention_rate=("top_term_retained", "mean"),
        mean_membership_jaccard=("membership_jaccard", "mean"),
        mean_cluster_term_jaccard=("cluster_term_jaccard", "mean"),
    )
    .reset_index()
)
retained_top_term_summary["classification"] = np.where(
    retained_top_term_summary["top_term_retention_rate"] >= STABLE_RETENTION_THRESHOLD,
    "stable",
    "unstable",
)
retained_top_term_summary = retained_top_term_summary.sort_values(["ref_cluster_id", "noise_frac"])
retained_top_term_summary

,ref_cluster_id,noise_frac,ref_top_term,ref_n_genes,n_replicates,top_term_retention_rate,mean_membership_jaccard,mean_cluster_term_jaccard,classification
0,1,0.05,GPI anchor biosynthetic process,148,5,1.0,0.808861,0.591124,stable
1,1,0.10,GPI anchor biosynthetic process,148,5,1.0,0.763182,0.681426,stable
2,1,0.20,GPI anchor biosynthetic process,148,5,1.0,0.711895,0.619598,stable
3,1,0.30,GPI anchor biosynthetic process,148,5,1.0,0.655018,0.586753,stable
4,1,0.50,GPI anchor biosynthetic process,148,5,1.0,0.784751,0.726070,stable
5,2,0.05,vesicle-mediated transport,92,5,1.0,0.844598,0.812182,stable
6,2,0.10,vesicle-mediated transport,92,5,1.0,0.831899,0.812766,stable
7,2,0.20,vesicle-mediated transport,92,5,1.0,0.788796,0.803338,stable
8,2,0.30,vesicle-mediated transport,92,5,1.0,0.792535,0.812766,stable
9,2,0.50,vesicle-mediated transport,92,5,1.0,0.799678,0.818085,stable


## 9. Stable vs unstable annotation classification (overall, per cluster)

In [11]:
def classify_cluster(sub: pd.DataFrame) -> str:
    mild = sub.loc[sub["noise_frac"] == NOISE_FRACS[0], "top_term_retention_rate"]
    moderate = sub.loc[sub["noise_frac"] <= MODERATE_NOISE_CEILING, "top_term_retention_rate"]
    mild_ok = bool((mild >= STABLE_RETENTION_THRESHOLD).all()) if len(mild) else False
    moderate_ok = bool((moderate >= STABLE_RETENTION_THRESHOLD).all()) if len(moderate) else False
    if moderate_ok:
        return "core_stable"
    if mild_ok:
        return "narrow_operating_range"
    return "unstable"


overall_rows = []
for ref_cid, sub in retained_top_term_summary.groupby("ref_cluster_id"):
    overall_rows.append(
        {
            "ref_cluster_id": ref_cid,
            "ref_top_term": sub["ref_top_term"].iloc[0],
            "ref_n_genes": int(sub["ref_n_genes"].iloc[0]),
            "retention_rate_mild_0.05": float(
                sub.loc[sub["noise_frac"] == NOISE_FRACS[0], "top_term_retention_rate"].iloc[0]
            ),
            "retention_rate_moderate_0.20": float(
                sub.loc[sub["noise_frac"] == 0.20, "top_term_retention_rate"].iloc[0]
            ),
            "retention_rate_aggressive_0.50": float(
                sub.loc[sub["noise_frac"] == 0.50, "top_term_retention_rate"].iloc[0]
            ),
            "classification": classify_cluster(sub),
        }
    )

stable_vs_unstable_annotations = pd.DataFrame(overall_rows).sort_values(
    ["classification", "ref_cluster_id"]
)
stable_vs_unstable_annotations

,ref_cluster_id,ref_top_term,ref_n_genes,retention_rate_mild_0.05,retention_rate_moderate_0.20,retention_rate_aggressive_0.50,classification
0,1,GPI anchor biosynthetic process,148,1.0,1.0,1.0,core_stable
1,2,vesicle-mediated transport,92,1.0,1.0,1.0,core_stable
2,3,"mRNA splicing, via spliceosome",263,1.0,1.0,1.0,core_stable
3,4,cytoplasmic translation,358,1.0,0.8,0.8,core_stable
4,5,mitochondrial respiratory chain complex IV ass...,78,1.0,1.0,1.0,core_stable
5,6,DNA replication,52,1.0,1.0,1.0,core_stable
6,7,cell division,62,1.0,1.0,1.0,core_stable


## 10. Write tables

In [12]:
TABLES = {
    "per_replicate_cluster_raw.csv": per_replicate_cluster_raw,
    "per_replicate_global_raw.csv": per_replicate_global_raw,
    "stability_summary.csv": stability_summary,
    "retained_top_term_summary.csv": retained_top_term_summary,
    "stable_vs_unstable_annotations.csv": stable_vs_unstable_annotations,
}
table_paths = {}
for name, df in TABLES.items():
    out_path = NB_TABLES_DIR / name
    df.to_csv(out_path, index=False)
    table_paths[name] = str(out_path.relative_to(layout["repo_root"]))
    print(f"wrote {out_path}  ({len(df)} rows)")

wrote /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/10_noise_robustness_gi_pcc/per_replicate_cluster_raw.csv  (175 rows)
wrote /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/10_noise_robustness_gi_pcc/per_replicate_global_raw.csv  (25 rows)
wrote /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/10_noise_robustness_gi_pcc/stability_summary.csv  (6 rows)
wrote /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/10_noise_robustness_gi_pcc/retained_top_term_summary.csv  (35 rows)
wrote /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/10_noise_robustness_gi_pcc/stable_vs_unstable_a

## 11. Stability plots

Categorical colors use the validated 8-hue palette (fixed order, cluster identity fixed across panels); the retention heatmap uses the validated single-hue sequential blue ramp. Both from the dataviz skill's reference palette (`references/palette.md`).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

CATEGORICAL_PALETTE = [
    "#2a78d6",  # blue
    "#eb6834",  # orange
    "#1baf7a",  # aqua
    "#eda100",  # yellow
    "#e87ba4",  # magenta
    "#008300",  # green
    "#4a3aa7",  # violet
    "#e34948",  # red
]
INK_PRIMARY = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRIDLINE = "#e1e0d9"
SEQUENTIAL_BLUE = LinearSegmentedColormap.from_list(
    "sequential_blue",
    ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"],
)

ref_cluster_ids = sorted(reference_cluster_to_labels.keys())
cluster_colors = {
    cid: CATEGORICAL_PALETTE[i % len(CATEGORICAL_PALETTE)] for i, cid in enumerate(ref_cluster_ids)
}
# Full, untruncated cluster labels -- shared by the legend below and reused verbatim
# wherever a cluster identity needs to be displayed. No mid-word truncation: this is
# the single source of legend text, so there is nothing else to keep in sync.
cluster_legend_labels = {
    ref_cid: f"C{ref_cid}: {reference_top_term_by_cluster[ref_cid]}" for ref_cid in ref_cluster_ids
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax in axes:
    ax.set_facecolor("#fcfcfb")
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)
    for spine in ("left", "bottom"):
        ax.spines[spine].set_color(GRIDLINE)
    ax.tick_params(colors=INK_MUTED, labelsize=9)
    ax.grid(axis="y", color=GRIDLINE, linewidth=0.8, zorder=0)

# Panel A: top-term retention rate vs noise, one line per reference cluster.
ax = axes[0]
for ref_cid in ref_cluster_ids:
    sub = retained_top_term_summary[
        retained_top_term_summary["ref_cluster_id"] == ref_cid
    ].sort_values("noise_frac")
    ax.plot(
        sub["noise_frac"],
        sub["top_term_retention_rate"],
        marker="o",
        markersize=6,
        linewidth=2,
        color=cluster_colors[ref_cid],
        label=cluster_legend_labels[ref_cid],
        zorder=3,
    )
mean_curve = retained_top_term_summary.groupby("noise_frac")["top_term_retention_rate"].mean()
(mean_line,) = ax.plot(
    mean_curve.index,
    mean_curve.values,
    color=INK_PRIMARY,
    linewidth=2.5,
    linestyle="--",
    label="Mean (all clusters)",
    zorder=4,
)
ax.axhline(STABLE_RETENTION_THRESHOLD, color=INK_MUTED, linewidth=1, linestyle=":", zorder=2)
ax.set_xlabel("Noise level (fraction of off-diagonal std)", color=INK_SECONDARY, fontsize=10)
ax.set_ylabel("Top-term retention rate", color=INK_SECONDARY, fontsize=10)
ax.set_ylim(-0.05, 1.05)
ax.set_title("A. Headline-annotation retention", color=INK_PRIMARY, fontsize=11, loc="left")

# Panel B: cluster gene-membership Jaccard vs noise, one line per reference cluster.
ax = axes[1]
for ref_cid in ref_cluster_ids:
    sub = per_replicate_cluster_raw[per_replicate_cluster_raw["ref_cluster_id"] == ref_cid]
    means = sub.groupby("noise_frac")["membership_jaccard"].mean()
    ax.plot(
        means.index,
        means.values,
        marker="o",
        markersize=6,
        linewidth=2,
        color=cluster_colors[ref_cid],
        zorder=3,
    )
ax.set_xlabel("Noise level (fraction of off-diagonal std)", color=INK_SECONDARY, fontsize=10)
ax.set_ylabel(
    "Mean gene-membership Jaccard\n(best-matched cluster)", color=INK_SECONDARY, fontsize=10
)
ax.set_ylim(-0.05, 1.05)
ax.set_title("B. Cluster membership stability", color=INK_PRIMARY, fontsize=11, loc="left")

# Panel C: global significant-term-set Jaccard vs noise (single aggregate series).
ax = axes[2]
means = per_replicate_global_raw.groupby("noise_frac")["global_sig_term_jaccard"].mean()
sds = per_replicate_global_raw.groupby("noise_frac")["global_sig_term_jaccard"].std()
ax.errorbar(
    means.index,
    means.values,
    yerr=sds.values,
    marker="o",
    markersize=6,
    linewidth=2,
    color=CATEGORICAL_PALETTE[0],
    ecolor=CATEGORICAL_PALETTE[0],
    elinewidth=1,
    capsize=3,
    zorder=3,
)
ax.set_xlabel("Noise level (fraction of off-diagonal std)", color=INK_SECONDARY, fontsize=10)
ax.set_ylabel(
    "Global significant-term Jaccard\n(cluster-agnostic)", color=INK_SECONDARY, fontsize=10
)
ax.set_ylim(-0.05, 1.05)
ax.set_title("C. Cluster-agnostic term retention", color=INK_PRIMARY, fontsize=11, loc="left")

fig.suptitle(
    "Noise robustness of yeast GI-PCC annotations (fig_1.ipynb reference; verified HiMaLAYAS v0.0.15)",
    color=INK_PRIMARY,
    fontsize=12,
    y=1.05,
)
# Single shared legend below all three panels, at a readable font size, with full
# (non-truncated) cluster labels. Panels A and B both use `cluster_colors`, so one
# legend correctly documents both; Panel C is a single aggregate series with no
# per-cluster color and is unaffected.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.14),
    ncol=4,
    fontsize=9.5,
    frameon=False,
    labelcolor=INK_SECONDARY,
    columnspacing=1.4,
    handlelength=1.6,
)
fig.tight_layout()
stability_curves_path = NB_FIGURES_DIR / "stability_curves.png"
fig.savefig(stability_curves_path, dpi=200, bbox_inches="tight", facecolor="#fcfcfb")
plt.show()
print(f"Saved {stability_curves_path}")

In [ ]:
noise_levels_for_heatmap = [0.0] + NOISE_FRACS
heat_data = np.full((len(ref_cluster_ids), len(noise_levels_for_heatmap)), np.nan)
row_labels = []
for i, ref_cid in enumerate(ref_cluster_ids):
    term = reference_top_term_by_cluster.get(ref_cid, "")
    # Full term shown when it fits; otherwise truncated at a word boundary with an
    # explicit ellipsis, never mid-word, so a shortened label is visibly a shortened
    # label rather than looking like the complete term.
    if len(term) <= 30:
        label_term = term
    else:
        label_term = term[:30].rsplit(" ", 1)[0] + "…"
    row_labels.append(f"C{ref_cid}: {label_term}")
    for j, frac in enumerate(noise_levels_for_heatmap):
        if frac == 0.0:
            heat_data[i, j] = 1.0
        else:
            row = retained_top_term_summary[
                (retained_top_term_summary["ref_cluster_id"] == ref_cid)
                & (retained_top_term_summary["noise_frac"] == frac)
            ]
            heat_data[i, j] = row["top_term_retention_rate"].iloc[0]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
im = ax.imshow(heat_data, cmap=SEQUENTIAL_BLUE, vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(noise_levels_for_heatmap)))
ax.set_xticklabels(
    [f"{f:.2f}" if f > 0 else "0.00\n(reference)" for f in noise_levels_for_heatmap],
    fontsize=9,
    color=INK_SECONDARY,
)
ax.set_yticks(range(len(ref_cluster_ids)))
ax.set_yticklabels(row_labels, fontsize=9, color=INK_SECONDARY)
ax.set_xlabel("Noise level (fraction of off-diagonal std)", color=INK_SECONDARY, fontsize=10)
ax.set_title(
    "Top-term retention rate by reference cluster and noise level",
    color=INK_PRIMARY,
    fontsize=11,
    loc="left",
)
for i in range(heat_data.shape[0]):
    for j in range(heat_data.shape[1]):
        val = heat_data[i, j]
        text_color = "white" if val > 0.55 else INK_PRIMARY
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8.5, color=text_color)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
cbar.set_label("Retention rate", color=INK_SECONDARY, fontsize=9)
cbar.ax.tick_params(colors=INK_MUTED, labelsize=8)
fig.tight_layout()
retention_heatmap_path = NB_FIGURES_DIR / "retention_heatmap.png"
fig.savefig(retention_heatmap_path, dpi=200, bbox_inches="tight", facecolor="#fcfcfb")
plt.show()
print(f"Saved {retention_heatmap_path}")

## 12. Go / Conditional Go / Concern decision

Decision rule (frozen plan Section 4), computed from the classification table above, not asserted:

- **GO**: no reference cluster is flatly `unstable` at the mildest tested noise, and at least 70% of clusters remain `core_stable` through moderate noise (frac <= 0.20).
- **CONDITIONAL GO**: at most one cluster is flatly `unstable`, and every other cluster is at least `narrow_operating_range` or better.
- **CONCERN**: two or more clusters are flatly `unstable` at mild noise, or the global cluster-agnostic term-set Jaccard falls below 0.5 at the mildest tested level.

In [15]:
import math

n_clusters = len(stable_vs_unstable_annotations)
counts = stable_vs_unstable_annotations["classification"].value_counts().to_dict()
n_core_stable = counts.get("core_stable", 0)
n_narrow = counts.get("narrow_operating_range", 0)
n_unstable = counts.get("unstable", 0)

mild_global_jaccard = float(
    stability_summary.loc[
        stability_summary["noise_frac"] == NOISE_FRACS[0], "mean_global_sig_term_jaccard"
    ].iloc[0]
)

print(
    f"Cluster classification counts (of {n_clusters}): "
    f"core_stable={n_core_stable}, narrow_operating_range={n_narrow}, unstable={n_unstable}"
)
print(
    f"Global cluster-agnostic significant-term Jaccard at mildest noise "
    f"(frac={NOISE_FRACS[0]}): {mild_global_jaccard:.3f}"
)

if n_unstable == 0 and n_core_stable >= math.ceil(0.7 * n_clusters):
    decision = "GO"
elif n_unstable <= 1 and (n_core_stable + n_narrow) == n_clusters:
    decision = "CONDITIONAL_GO"
else:
    decision = "CONCERN"

if mild_global_jaccard < 0.5:
    decision = "CONCERN"

print(f"\nDECISION: {decision}")

Cluster classification counts (of 7): core_stable=7, narrow_operating_range=0, unstable=0
Global cluster-agnostic significant-term Jaccard at mildest noise (frac=0.05): 0.831

DECISION: GO


## 13. Assemble and write manifest

In [16]:
flags = list(input_hash_flags)
if activation is None:
    flags.append(f"HIMALAYAS ACTIVATION FAILED: {activation_error}")

if decision == "GO":
    next_action = (
        "Proceed to 11_depth_threshold_sensitivity.ipynb. Headline functional annotations "
        f"({n_core_stable}/{n_clusters} reference clusters) are robust to moderate matrix "
        "perturbation (frac <= 0.20); no annotation is flatly unstable at mild noise. This is "
        "annotation retention, not unchanged cluster membership -- gene-membership Jaccard "
        "with the reference clusters declines with noise even where headline annotations hold."
    )
elif decision == "CONDITIONAL_GO":
    unstable_terms = stable_vs_unstable_annotations.loc[
        stable_vs_unstable_annotations["classification"] != "core_stable", "ref_top_term"
    ].tolist()
    next_action = (
        "Proceed to 11_depth_threshold_sensitivity.ipynb, but narrow claims to the "
        f"{n_core_stable}/{n_clusters} core-stable annotations; state the operating range "
        f"explicitly for the remainder ({unstable_terms}); frame HiMaLAYAS as an "
        "annotation/prioritization aid, not a universal discovery engine (frozen plan Section 4)."
    )
else:
    next_action = (
        "PAUSE manuscript polish. Mild noise reshuffles multiple key annotations "
        f"({n_unstable}/{n_clusters} clusters unstable already at frac={NOISE_FRACS[0]}, "
        f"global term-set Jaccard={mild_global_jaccard:.3f}). Reconsider claims, figures, or "
        "venue before investing further time (frozen plan Section 4, No-Go response)."
    )

membership_jaccard_at_mild = float(
    stability_summary.loc[
        stability_summary["noise_frac"] == NOISE_FRACS[0], "mean_membership_jaccard"
    ].iloc[0]
)
membership_jaccard_at_moderate = float(
    stability_summary.loc[
        stability_summary["noise_frac"] == MODERATE_NOISE_CEILING, "mean_membership_jaccard"
    ].iloc[0]
)
membership_jaccard_min_overall = float(stability_summary["min_membership_jaccard"].min())

manifest = {
    "notebook": "10_noise_robustness_gi_pcc.ipynb",
    "status": "COMPLETE" if activation is not None else "BLOCKED",
    "decision": decision,
    "next_action": next_action,
    "generated_at_utc": ru.utc_timestamp(),
    "repo_root": str(layout["repo_root"]),
    "himalayas_source": activation if activation is not None else {"error": activation_error},
    "himalayas_diagnostics_at_start": himalayas_diag_before,
    "input_files": {
        "gi_pcc_matrix": {
            "path": "data/yeast/gi_pcc_sampled.tsv",
            "sha256": current_hashes["data/yeast/gi_pcc_sampled.tsv"],
            "matches_nb00_record": current_hashes["data/yeast/gi_pcc_sampled.tsv"]
            == recorded_hashes.get("data/yeast/gi_pcc_sampled.tsv"),
        },
        "go_bp_annotations": {
            "path": "data/yeast/go_bp_name_to_orfs.json",
            "sha256": current_hashes["data/yeast/go_bp_name_to_orfs.json"],
            "matches_nb00_record": current_hashes["data/yeast/go_bp_name_to_orfs.json"]
            == recorded_hashes.get("data/yeast/go_bp_name_to_orfs.json"),
        },
    },
    "reference_config": {
        **REFERENCE_PARAMS,
        "min_overlap": MIN_OVERLAP,
        "qval_cutoff": QVAL_CUTOFF,
    },
    "reference_reproduction_check": {
        "matches_nb01_baseline": bool(reproduction_matches),
        "results_rows": len(reference_results.df),
        "results_sig_rows": len(reference_results_sig.df),
        "n_clusters": len(reference_cluster_labels),
    },
    "noise_design": {
        "offdiag_std": offdiag_std,
        "noise_fracs": NOISE_FRACS,
        "n_seeds": N_SEEDS,
        "replicate_seeds": REPLICATE_SEEDS,
        "default_random_seed": DEFAULT_RANDOM_SEED,
        "stable_retention_threshold": STABLE_RETENTION_THRESHOLD,
        "moderate_noise_ceiling": MODERATE_NOISE_CEILING,
        "noise_model": (
            "perturbed = original + scale * (E + E.T) / sqrt(2), E ~ iid N(0,1), "
            "scale = noise_frac * offdiag_std; symmetric noise only, does not alter "
            "whatever asymmetry the original matrix already has."
        ),
    },
    "cluster_classification_counts": {
        "core_stable": n_core_stable,
        "narrow_operating_range": n_narrow,
        "unstable": n_unstable,
        "total": n_clusters,
    },
    "flags": flags,
    "outputs": {
        "tables": table_paths,
        "figures": {
            "stability_curves.png": str(stability_curves_path.relative_to(layout["repo_root"])),
            "retention_heatmap.png": str(retention_heatmap_path.relative_to(layout["repo_root"])),
        },
    },
    "interpretation": (
        f"Decision={decision}. Claim: headline functional annotations are robust to moderate "
        f"matrix perturbation. {n_core_stable}/{n_clusters} reference-cluster headline "
        f"annotations remain stable (>= {STABLE_RETENTION_THRESHOLD:.0%} retention) through "
        f"moderate noise (frac <= {MODERATE_NOISE_CEILING}); {n_narrow} are stable only in a "
        f"narrow operating range; {n_unstable} are unstable already at mild noise "
        f"(frac={NOISE_FRACS[0]}). This does not imply unchanged cluster membership: mean "
        f"gene-membership Jaccard between each reference cluster and its best-matched "
        f"perturbed cluster falls from {membership_jaccard_at_mild:.2f} at frac="
        f"{NOISE_FRACS[0]} to {membership_jaccard_at_moderate:.2f} at frac="
        f"{MODERATE_NOISE_CEILING} (individual replicates as low as "
        f"{membership_jaccard_min_overall:.2f}), even as headline-term retention stays at or "
        f"above {STABLE_RETENTION_THRESHOLD:.0%}. Noise was applied uniformly to the full "
        "matrix, including diagonal entries (no special-casing). Cluster-agnostic global "
        f"significant-term-set Jaccard at mildest noise: {mild_global_jaccard:.3f}. See "
        "stable_vs_unstable_annotations.csv for the per-annotation breakdown and "
        "stability_curves.png / retention_heatmap.png for the full noise-dependence."
    ),
}

manifest_path = layout["manifests_dir"] / f"{NB_ID}_manifest.json"
ru.write_manifest(manifest, manifest_path)
print(f"Manifest written to: {manifest_path}")

reloaded = json.loads(manifest_path.read_text())
assert reloaded == json.loads(json.dumps(manifest, default=str)), "manifest round-trip mismatch"
print("Manifest JSON round-trip verified.")

Manifest written to: /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests/10_noise_robustness_gi_pcc_manifest.json
Manifest JSON round-trip verified.


## 14. Readiness summary

In [17]:
print("=" * 72)
print(f"DECISION -- {NB_ID}")
print("=" * 72)
print(f"status:      {manifest['status']}")
print(f"decision:    {manifest['decision']}")
print(f"next_action: {manifest['next_action']}")
print()
print(
    f"HiMaLAYAS source: {activation['activated_version'] if activation else 'N/A'} "
    f"(verified tag {PINNED_TAG}, not the active dev install)"
)
print(f"Reference reproduction matches notebook 01: {reproduction_matches}")
print()
print("Claim: headline functional annotations are robust to moderate matrix perturbation")
print("(this is annotation retention, not unchanged cluster membership -- gene-membership")
print(
    f"Jaccard falls to ~{membership_jaccard_at_moderate:.2f} by frac={MODERATE_NOISE_CEILING} "
    "even where headline terms are retained)."
)
print()
print("Per-cluster classification:")
display(
    stable_vs_unstable_annotations[
        [
            "ref_cluster_id",
            "ref_top_term",
            "classification",
            "retention_rate_mild_0.05",
            "retention_rate_moderate_0.20",
            "retention_rate_aggressive_0.50",
        ]
    ]
)
print()
print(f"Manifest:  {manifest_path}")
print(f"Tables:    {NB_TABLES_DIR}")
print(f"Figures:   {NB_FIGURES_DIR}")

DECISION -- 10_noise_robustness_gi_pcc
status:      COMPLETE
decision:    GO
next_action: Proceed to 11_depth_threshold_sensitivity.ipynb. Headline functional annotations (7/7 reference clusters) are robust to moderate matrix perturbation (frac <= 0.20); no annotation is flatly unstable at mild noise. This is annotation retention, not unchanged cluster membership -- gene-membership Jaccard with the reference clusters declines with noise even where headline annotations hold.

HiMaLAYAS source: 0.0.15 (verified tag v0.0.15, not the active dev install)
Reference reproduction matches notebook 01: True

Claim: headline functional annotations are robust to moderate matrix perturbation
(this is annotation retention, not unchanged cluster membership -- gene-membership
Jaccard falls to ~0.67 by frac=0.2 even where headline terms are retained).

Per-cluster classification:


,ref_cluster_id,ref_top_term,classification,retention_rate_mild_0.05,retention_rate_moderate_0.20,retention_rate_aggressive_0.50
0,1,GPI anchor biosynthetic process,core_stable,1.0,1.0,1.0
1,2,vesicle-mediated transport,core_stable,1.0,1.0,1.0
2,3,"mRNA splicing, via spliceosome",core_stable,1.0,1.0,1.0
3,4,cytoplasmic translation,core_stable,1.0,0.8,0.8
4,5,mitochondrial respiratory chain complex IV ass...,core_stable,1.0,1.0,1.0
5,6,DNA replication,core_stable,1.0,1.0,1.0
6,7,cell division,core_stable,1.0,1.0,1.0



Manifest:  /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/manifests/10_noise_robustness_gi_pcc_manifest.json
Tables:    /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/tables/10_noise_robustness_gi_pcc
Figures:   /Users/irahorecka/Desktop/harddrive_desktop/PhD/University of Toronto/Rost Lab/GitHub/himalayas-publication/revision/outputs/figures/10_noise_robustness_gi_pcc
